In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

Adult Census Income Dataset Description

| Column | Description | Values / Type |
|--------|-------------|---------------|
| **age** | Age of individual (continuous) | 17-90 years |
| **workclass** | Employment type | Private, Self-emp-not-inc, Self-emp-inc, Federal-gov, Local-gov, State-gov, Without-pay, Never-worked |
| **fnlwgt** | Final weight - population weighting factor (number of people the census believes the entry represents) | Integer (large numbers) |
| **education** | Highest education level attained | Bachelors, HS-grad, 11th, Masters, Assoc-acdm, Assoc-voc, Doctorate, Prof-school, Some-college, 10th, 9th, 7th-8th, 5th-6th, 1st-4th, Preschool |
| **education-num** | Number of years of education | 1-16 years |
| **marital-status** | Marital status | Married-civ-spouse, Divorced, Never-married, Separated, Widowed, Married-spouse-absent, Married-AF-spouse |
| **occupation** | Job type | Adm-clerical, Exec-managerial, Handlers-cleaners, Prof-specialty, Other-service, Sales, Craft-repair, Transport-moving, Farming-fishing, Machine-op-inspct, Tech-support, Protective-serv, Armed-Forces, Priv-house-serv |
| **relationship** | Family relationship status | Wife, Husband, Own-child, Not-in-family, Other-relative, Unmarried |
| **race** | Race category | White, Black, Asian-Pac-Islander, Amer-Indian-Eskimo, Other |
| **sex** | Gender | Male, Female |
| **capital-gain** | Capital gains (income from investments) | 0 to large positive numbers |
| **capital-loss** | Capital losses (losses from investments) | 0 to large positive numbers |
| **hours-per-week** | Hours worked per week | 1-99 hours |
| **native-country** | Country of origin | United-States, Cuba, Mexico, Philippines, etc. |
| **income** | **Target variable** — Income level | <=50K, >50K |

In [2]:
column_names = [
    'age',
    'workclass',
    'fnlwgt',
    'education',
    'education_num',
    'marital_status',
    'occupation',
    'relationship',
    'race',
    'sex',
    'capital_gain',
    'capital_loss',
    'hours_per_week',
    'native_country',
    'income'
]

In [3]:
data = pd.read_csv('../datasets/adults.csv', names=column_names, skipinitialspace=True)
data.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education_num   32561 non-null  int64 
 5   marital_status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital_gain    32561 non-null  int64 
 11  capital_loss    32561 non-null  int64 
 12  hours_per_week  32561 non-null  int64 
 13  native_country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


### 1. Create socioeconomic classes and analyze wealth distribution patterns.

Create Wealth Index

Combine the following features into a single `wealth_index`:

| Feature | Weight / Method |
|---------|-----------------|
| `capital-gain` | Normalized (0-1) |
| `capital-loss` | Normalized (0-1) |
| `education-num` | Normalized (0-1) |

**Formula:**

```python
wealth_index = (normalized_capital_gain + normalized_capital_loss + normalized_education_num) / 3
```

Stratify individuals into **4 socioeconomic classes** using quartiles of `wealth_index`:

| Class | Quartile Range |
|-------|----------------|
| **Lower** | Q1 (0-25%) |
| **Lower-Middle** | Q2 (25-50%) |
| **Upper-Middle** | Q3 (50-75%) |
| **Upper** | Q4 (75-100%) |



For each socioeconomic class, calculate the following metrics:

| Metric | Description |
|--------|-------------|
| **Mean hours-per-week** | Average hours worked per week |
| **Percentage with income >50K** | % of individuals earning above $50K |
| **Most common occupation** | Most frequent occupation in the class |
| **Education distribution** | Top 3 education levels in the class |

Find the socioeconomic class with the **highest representation** in the `>50K` income group.

In [5]:
scaler_cg = MinMaxScaler()
scaler_cl = MinMaxScaler()
scaler_en = MinMaxScaler()

cg_normalized = scaler_cg.fit_transform(data[['capital_gain']]).flatten()
cl_normalized = scaler_cl.fit_transform(data[['capital_loss']]).flatten()
en_normalized = scaler_en.fit_transform(data[['education_num']]).flatten()

data['wealth_index'] = (cg_normalized + cl_normalized + en_normalized) / 3

In [6]:
data[['education_num', 'capital_gain', 'capital_loss', 'wealth_index']].head()

,education_num,capital_gain,capital_loss,wealth_index
0,13,2174,0,0.273913
1,13,0,0,0.266667
2,9,0,0,0.177778
3,7,0,0,0.133333
4,13,0,0,0.266667


In [7]:
quartile = [0, .25, .5, .75, 1.]
labels = ['lower', 'lower-middle', 'upper-middle', 'upper']
data['wealth_class'] = pd.qcut(x=data['wealth_index'], q=[0, .25, .5, .75, 1.], labels=labels)
data['class_index'] = data['wealth_class'].map({
    'lower': 1,
    'lower-middle': 2,
    'upper-middle': 3,
    'upper': 4
})

In [8]:
data[['education_num', 'capital_gain', 'capital_loss', 'wealth_index', 'wealth_class', 'class_index']].head()

,education_num,capital_gain,capital_loss,wealth_index,wealth_class,class_index
0,13,2174,0,0.273913,upper,4
1,13,0,0,0.266667,upper-middle,3
2,9,0,0,0.177778,lower,1
3,7,0,0,0.133333,lower,1
4,13,0,0,0.266667,upper-middle,3


In [9]:
def wealth_stats(group):
    count_with_income_gt_50k = group[group['income']=='>50K'].shape[0]
    count_total = group.shape[0]
    
    mode_occupation = group['occupation'].mode()
    most_common_occupation = mode_occupation[0] if not mode_occupation.empty else np.nan

    education_counts = group['education'].value_counts()
    top_3_education = education_counts.head(3).index.tolist()
        
    pct_with_income_gt_50k = round((group['income']=='>50K').mean() * 100, 2)
    
    return pd.Series(
        {
            'mean_hours_per_weak': group['hours_per_week'].mean().round(2),
            'percentage_with_income_gt_50k': round(count_with_income_gt_50k * 100 /count_total, 2),
            'most_common_occupation': most_common_occupation,
            'top_3_education': top_3_education,
            'percentage_with_income_gt_50k_2': pct_with_income_gt_50k
        }
    )
    
data.groupby(['wealth_class'], observed=True).apply(wealth_stats, include_groups=False).reset_index()

,wealth_class,mean_hours_per_weak,percentage_with_income_gt_50k,most_common_occupation,top_3_education,percentage_with_income_gt_50k_2
0,lower,39.27,10.61,Craft-repair,"[HS-grad, 11th, 10th]",10.61
1,lower-middle,38.62,15.86,Adm-clerical,"[Some-college, HS-grad, 7th-8th]",15.86
2,upper-middle,41.81,33.17,Prof-specialty,"[Bachelors, Assoc-voc, Assoc-acdm]",33.17
3,upper,44.43,61.56,Prof-specialty,"[Masters, Bachelors, Prof-school]",61.56


In [10]:
occupation = data.groupby(['wealth_class', 'occupation'], observed=True).size().reset_index().rename(columns={0: 'count'})
occupation['rank'] = occupation.groupby(['wealth_class'], observed=True)['count'].rank(method='dense', ascending=False)
most_common_occupation = occupation[occupation['rank']==1]
most_common_occupation

,wealth_class,occupation,count,rank
3,lower,Craft-repair,2337,1.0
16,lower-middle,Adm-clerical,1225,1.0
40,upper-middle,Prof-specialty,1581,1.0
55,upper,Prof-specialty,1924,1.0


In [11]:
occupation = data.groupby(['wealth_class', 'education'], observed=True).size().reset_index().rename(columns={0: 'count'})
occupation['rank'] = occupation.groupby(['wealth_class'], observed=True)['count'].rank(method='dense', ascending=False)
occupation[occupation['rank']<=3]

,wealth_class,education,count,rank
0,lower,10th,902,3.0
1,lower,11th,1139,2.0
7,lower,HS-grad,9415,1.0
14,lower-middle,7th-8th,11,3.0
15,lower-middle,HS-grad,448,2.0
16,lower-middle,Some-college,6533,1.0
24,upper-middle,Assoc-acdm,972,3.0
25,upper-middle,Assoc-voc,1308,2.0
26,upper-middle,Bachelors,4384,1.0
36,upper,Bachelors,971,2.0


### 2. Investigate the relationship between hours worked, education, and income to find anomalies.

Step 1: Create Hours Category

Create `hours_category` based on weekly hours:

| Category | Hours Range |
|----------|-------------|
| **Part-time** | < 30 hours |
| **Full-time** | 30 - 45 hours |
| **Overtime** | 46 - 60 hours |
| **Extreme** | > 60 hours |

Step 2: Calculate Metrics per Occupation

For each occupation, calculate the following:

| Metric | Description |
|--------|-------------|
| **Average hours-per-week** | Mean hours worked per week |
| **Average education-num** | Mean years of education |
| **Income >50K percentage** | % of individuals earning above $50K |

Step 3: Create Efficiency Score

$$
\text{efficiency\_score} = \frac{\text{income\_level}}{\text{hours\_per\_week}}
$$

Where:
- **income_level** = `1` if income >50K, else `0`

Step 4: Identify High & Low Efficiency Roles

| Category | Pattern |
|----------|---------|
| **High-Efficiency** | Low hours, high income |
| **Low-Efficiency** | High hours, low income |

Step 5: Flag High Efficiency Occupations

Flag occupations with **efficiency_score > 0.02** as `'High Efficiency'`.


In [12]:
# step 1
conditions = {
    'part_time': data['hours_per_week'] < 30,
    'full_time': data['hours_per_week'].between(30, 45),
    'overtime': data['hours_per_week'].between(46, 60),
    'extreme': data['hours_per_week'] > 60,    
}
data['hours_category'] = np.select(condlist=list(conditions.values()), choicelist=list(conditions.keys()), default='unknown')

In [13]:
# step 2
def occupation_stats(group):
    
    
    return pd.Series(
        {
            'mean_hours_worked_per_week': group['hours_per_week'].mean().round(2),
            'mean_years_of_education': group['education_num'].mean().round(2),
            '%_of_individuals_earning_above_50K': round((group['income'] == '>50K').mean() * 100, 2),
        }
    )

occupation = data.groupby(['occupation'], observed=True).apply(occupation_stats, include_groups=False)
occupation

,mean_hours_worked_per_week,mean_years_of_education,%_of_individuals_earning_above_50K
occupation,,,
?,31.91,9.25,10.36
Adm-clerical,37.56,10.11,13.45
Armed-Forces,40.67,10.11,11.11
Craft-repair,42.30,9.11,22.66
Exec-managerial,44.99,11.45,48.40
Farming-fishing,46.99,8.61,11.57
Handlers-cleaners,37.95,8.51,6.28
Machine-op-inspct,40.76,8.49,12.49
Other-service,34.70,8.78,4.16


In [14]:
occupation_agg = data.groupby('occupation').agg(
    mean_hours_worked_per_week=('hours_per_week', 'mean'),
    mean_years_of_education=('education_num', 'mean'),
    pct_income_gt_50k=('income', lambda x: (x == '>50K').mean() * 100)
).round(2)
occupation_agg

,mean_hours_worked_per_week,mean_years_of_education,pct_income_gt_50k
occupation,,,
?,31.91,9.25,10.36
Adm-clerical,37.56,10.11,13.45
Armed-Forces,40.67,10.11,11.11
Craft-repair,42.30,9.11,22.66
Exec-managerial,44.99,11.45,48.40
Farming-fishing,46.99,8.61,11.57
Handlers-cleaners,37.95,8.51,6.28
Machine-op-inspct,40.76,8.49,12.49
Other-service,34.70,8.78,4.16


In [15]:
# step 3
data['income_level'] = (data['income'] == '>50K') * 1
data['efficiency_score'] = data['income_level'] / data['hours_per_week']

In [16]:
# step 4
data['efficiency_score_flag'] = np.where(data['efficiency_score']>0.02, 'High Efficiency', 'Standard')

In [17]:
data.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,...,hours_per_week,native_country,income,wealth_index,wealth_class,class_index,hours_category,income_level,efficiency_score,efficiency_score_flag
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,...,40,United-States,<=50K,0.273913,upper,4,full_time,0,0.0,Standard
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,...,13,United-States,<=50K,0.266667,upper-middle,3,part_time,0,0.0,Standard
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,...,40,United-States,<=50K,0.177778,lower,1,full_time,0,0.0,Standard
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,...,40,United-States,<=50K,0.133333,lower,1,full_time,0,0.0,Standard
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,...,40,Cuba,<=50K,0.266667,upper-middle,3,full_time,0,0.0,Standard


### 3. Analyze how immigrant status and education interact with income potential.

Step 1: Create `immigrant_status` column:

| Condition | Status |
|-----------|--------|
| `native-country` is `'United-States'` | `'Native'` |
| Otherwise | `'Immigrant'` |


Step 2: For each combination of `immigrant_status` and `education` calculate:

| Metric | Description |
|--------|-------------|
| **Income >50K Rate** | % of individuals earning above $50K |
| **Average Capital-Gain** | Mean capital gains |
| **Average Capital-Loss** | Mean capital losses |
| **Education Premium** | `(income_rate - average_income_rate) / average_income_rate` |


Step 3: Create a pivot table showing **income >50K rates** with:

- **Rows:** `education`
- **Columns:** `immigrant_status` (`Native` vs `Immigrant`)


Step 4: Find the **top 5 education levels** where **immigrants outperform natives** (highest positive difference).



In [18]:
# step 1
data['immigrant_status'] = np.where(data['native_country'] == 'United-States', 'Native', 'Immegrant')
data.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,...,native_country,income,wealth_index,wealth_class,class_index,hours_category,income_level,efficiency_score,efficiency_score_flag,immigrant_status
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,...,United-States,<=50K,0.273913,upper,4,full_time,0,0.0,Standard,Native
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,...,United-States,<=50K,0.266667,upper-middle,3,part_time,0,0.0,Standard,Native
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,...,United-States,<=50K,0.177778,lower,1,full_time,0,0.0,Standard,Native
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,...,United-States,<=50K,0.133333,lower,1,full_time,0,0.0,Standard,Native
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,...,Cuba,<=50K,0.266667,upper-middle,3,full_time,0,0.0,Standard,Immegrant


In [19]:
# step 2
income = data.groupby(['immigrant_status', 'education']).agg(
    pct_of_individuals_earning_above_50K = ('income', lambda x: (x == '>50K').mean() * 100),
    mean_capital_gains = ('capital_gain', 'mean'),
    mean_capintal_loss = ('capital_loss', 'mean')
).round(2)

overall_rate = (data['income'] == '>50K').mean() * 100

income['education_premium'] = (income['pct_of_individuals_earning_above_50K'] - overall_rate) * 100 / overall_rate
income = income.reset_index()
income

,immigrant_status,education,pct_of_individuals_earning_above_50K,mean_capital_gains,mean_capintal_loss,education_premium
0,Immegrant,10th,3.53,24.76,23.26,-85.341113
1,Immegrant,11th,4.63,440.24,47.92,-80.773188
2,Immegrant,12th,7.35,173.62,27.18,-69.477956
3,Immegrant,1st-4th,4.10,173.34,34.47,-82.974098
4,Immegrant,5th-6th,4.66,160.23,35.97,-80.648609
5,Immegrant,7th-8th,6.12,209.16,85.27,-74.585726
6,Immegrant,9th,3.36,98.64,28.32,-86.047065
7,Immegrant,Assoc-acdm,21.18,1636.22,103.45,-12.046680
8,Immegrant,Assoc-voc,26.88,514.42,108.33,11.623477
9,Immegrant,Bachelors,34.80,1310.87,107.95,44.512537


In [20]:
# step 3
income.pivot(
    index='education', 
    columns='immigrant_status', 
    values='pct_of_individuals_earning_above_50K'
)

immigrant_status,Immegrant,Native
education,,
10th,3.53,6.96
11th,4.63,5.15
12th,7.35,7.67
1st-4th,4.10,2.17
5th-6th,4.66,5.15
7th-8th,6.12,6.21
9th,3.36,5.82
Assoc-acdm,21.18,25.15
Assoc-voc,26.88,26.07


In [21]:
immegrant =data.pivot_table(
    index='education', 
    columns='immigrant_status', 
    values='income',
    aggfunc=lambda x: (x == '>50K').mean() * 100
)
immegrant

immigrant_status,Immegrant,Native
education,,
10th,3.529412,6.957547
11th,4.629630,5.154639
12th,7.352941,7.671233
1st-4th,4.098361,2.173913
5th-6th,4.661017,5.154639
7th-8th,6.122449,6.212425
9th,3.361345,5.822785
Assoc-acdm,21.176471,25.152749
Assoc-voc,26.881720,26.066718


In [22]:
# step 4
immegrant['diff'] = immegrant['Immegrant'] - immegrant['Native']
top_5_immigrant_advantage = immegrant.sort_values(by=['diff'], ascending=False).head(5)
top_5_immigrant_advantage

immigrant_status,Immegrant,Native,diff
education,,,
1st-4th,4.098361,2.173913,1.924448
Assoc-voc,26.881720,26.066718,0.815002
Preschool,0.000000,0.000000,0.000000
7th-8th,6.122449,6.212425,-0.089976
12th,7.352941,7.671233,-0.318292


### 4. Career-Life Stage Pattern Analysis

Use advanced grouping and window functions to analyze career-life stage patterns.

Step 1: Create `life_stage` based on age:

| Life Stage | Age Range |
|------------|-----------|
| **Young** | < 30 |
| **Mid-Career** | 30 - 50 |
| **Senior** | > 50 |


Step 2: For each `marital-status`, calculate **rolling (3-step) averages** over age:

| Metric | Description |
|--------|-------------|
| **Rolling Average of Education-num** | 3-step average of education level |
| **Rolling Average of Hours-per-week** | 3-step average of working hours |
| **Rolling Count of Income >50K** | 3-step count of high earners |



Step 3: For each individual:

$$
\text{progression\_score} = \left( \frac{\text{age}}{\text{education\_num}} \right) \times \left( \frac{\text{hours\_per\_week}}{40} \right)
$$

**Normalize** the score to 0-1 range.


Step 4: Find marital status groups with:

| Attribute | Description |
|-----------|-------------|
| **Highest Career Progression Trajectory** | Increasing `progression_score` with age |
| **Highest Income Stability** | Low variance in income >50K within the group |


In [25]:
# step 1
conditions = [data['age']<30, data['age'].between(30, 50), data['age']>50] 
labels = ['young', 'mid-career', 'senior']
data['life_stage'] = np.select(condlist=conditions, choicelist=labels, default='unknown')
data[['age', 'life_stage']].head()

,age,life_stage
0,39,mid-career
1,50,mid-career
2,38,mid-career
3,53,senior
4,28,young


In [51]:
# test: rolling average row by row
df_test = pd.DataFrame({
    'marital_status': ['Divorced'] * 10,
    'age': [22, 19, 19, 19, 20, 20, 21, 21, 21, 18],
    'education_num': [10, 12, 14, 11, 13, 15, 10, 12, 14, 16],
    'hours_per_week': [30, 35, 40, 38, 42, 45, 36, 40, 44, 50],
    'income': ['<=50K', '<=50K', '>50K', '<=50K', '>50K', '>50K', '<=50K', '<=50K', '>50K', '>50K']
})
df_test
# index: 3 --> (11 + 14 + 12)/3
# index: 8 --> (14 + 12 + 10)/3

,marital_status,age,education_num,hours_per_week,income
0,Divorced,22,10,30,<=50K
1,Divorced,19,12,35,<=50K
2,Divorced,19,14,40,>50K
3,Divorced,19,11,38,<=50K
4,Divorced,20,13,42,>50K
5,Divorced,20,15,45,>50K
6,Divorced,21,10,36,<=50K
7,Divorced,21,12,40,<=50K
8,Divorced,21,14,44,>50K
9,Divorced,18,16,50,>50K


In [52]:
df_test['income_numeric'] = (df_test['income'] == '>50K').astype(int)

df_test.groupby('marital_status').rolling(
    window=3,
    on='age',
    min_periods=1
).agg({
    'education_num': 'mean',
    'hours_per_week': 'mean',
    'income_numeric': 'sum'
}).round(1).reset_index()

,marital_status,age,education_num,hours_per_week,income_numeric
0,Divorced,22,10.0,30.0,0.0
1,Divorced,19,11.0,32.5,0.0
2,Divorced,19,12.0,35.0,1.0
3,Divorced,19,12.3,37.7,1.0
4,Divorced,20,12.7,40.0,2.0
5,Divorced,20,13.0,41.7,2.0
6,Divorced,21,12.7,41.0,2.0
7,Divorced,21,12.3,40.3,1.0
8,Divorced,21,12.0,40.0,1.0
9,Divorced,18,14.0,44.7,2.0


In [55]:
# step 2
data['income_numeric'] = (data['income'] == '>50K').astype(int)

rolling_agg = data.groupby(['marital_status']).rolling(
    window=3,
    on='age',
    min_periods=1
).agg({
    'education_num': 'mean',
    'hours_per_week': 'mean',
    'income_numeric': 'sum'
}).round(1).reset_index()
rolling_agg.head()

,marital_status,age,education_num,hours_per_week,income_numeric
0,Divorced,38,9.0,40.0,0.0
1,Divorced,43,11.5,42.5,1.0
2,Divorced,59,10.7,41.7,1.0
3,Divorced,39,10.7,55.0,1.0
4,Divorced,45,10.3,53.3,0.0


In [57]:
# step 3:
data['progression_score'] = (data['age'] / data['education_num']) * (data['hours_per_week'] / 40)
scaler_progression_score = MinMaxScaler()
data['progression_score_normalized'] = scaler_progression_score.fit_transform(data[['progression_score']])
data.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,...,class_index,hours_category,income_level,efficiency_score,efficiency_score_flag,immigrant_status,life_stage,income_numeric,progression_score,progression_score_normalized
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,...,4,full_time,0,0.0,Standard,Native,mid-career,0,3.000000,0.032706
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,...,3,part_time,0,0.0,Standard,Native,mid-career,0,1.250000,0.013249
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,...,1,full_time,0,0.0,Standard,Native,mid-career,0,4.222222,0.046295
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,...,1,full_time,0,0.0,Standard,Native,senior,0,7.571429,0.083533
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,...,3,full_time,0,0.0,Standard,Immegrant,young,0,2.153846,0.023299


### 5. Create sophisticated features and reduce dimensions while preserving information.


Step 1: Create New Features

| New Feature | Formula |
|-------------|---------|
| **wealth_potential** | `(capital_gain - capital_loss) * (education_num / 13)` |
| **work_intensity** | `hours_per_week * (1 / (education_num + 1))` |
| **social_mobility** | `(income_target - avg_income_by_occupation) / std_income_by_occupation` |
| **age_education_ratio** | `age / education_num` |
| **career_stage** | Categorize based on age and education |



Career Stage Classification

| Career Stage | Criteria |
|--------------|----------|
| **Early Bloomer** | Age < 35, Education > 13 |
| **Late Starter** | Age > 40, Education < 13 |
| **Steady Climber** | Age 35-50, Education 13-15 |
| **Plateaued** | Age > 50, Education < 13 |


Step 2: Feature Selection

- Calculate **correlation matrix** with target `income`
- Keep **top 5 features** with highest absolute correlation
- Create a **reduced dataset** with only these features + target


Step 3: Compare Performance

Compare predictive performance using **simple logistic regression** between:

- **Full dataset** (all features)
- **Reduced dataset** (top 5 features)


Step 4: One-Hot Encoding

Create one-hot encoding for:

- `workclass`
- `marital-status`
- `occupation` (top 5 most frequent categories, group others as `'Other'`)
